# Interpret checkpoint (single episode)

Load a **``.pt``** checkpoint and a snapshot **JSON**, build :class:`~env.game_simulator.GameSimulator`, then roll out **one** episode with the policy. The loop mirrors training rollouts: **``Categorical.sample``** on each hand-slot selection logit and on the play/discard head, using ``get_card_mask`` / ``mask_logits`` the same way as :func:`agent.ppo.compute_log_prob_and_entropy`, then ``GameSimulator.step`` / ``step_print``.

**Configure first:** use the **Run configuration** section (right after repo/mount): set ``AGENT_KIND``, ``CKPT_PATH``, ``SNAPSHOT_PATH``, and optional ``SEED`` / ``MAX_STEPS`` / ``DEVICE`` there.

**Policy:** ``AGENT_KIND`` is ``"full"`` (:class:`~agent.model.CombatPPOAgent`, e.g. ``combat_ppo_iter_*.pt``) or ``"minimal"`` (:class:`~agent.minimal_model.MinimalCombatPPOAgent`, e.g. ``minimal_combat_ppo_iter_*.pt``). Widths in the checkpoint (``d_model``, ``nhead``, …) come from the saved ``PPOConfig``; transformer **depth** must match the class defaults used when the checkpoint was trained—full model: ``CombatBackbone`` depths; minimal: ``MinimalCombatBackbone`` ``depth_hand``, ``depth_hd``, ``depth_hc`` (same as ``agent/TrainCombat.ipynb`` / ``agent/TrainCombatMinimal.ipynb`` unless you changed those constructors).

**Flow:** ``GameSimulator.from_json`` → loop: policy → ``step_print`` runs ``env.step`` and prints the snapshot after each step (and a one-line action summary).

**One environment only:** there is a single `BalatroEnv` behind `GameSimulator`—no `VectorEnv`, no parallel workers, and no multi-rollout batch. Evaluation is always one trajectory until terminal or `MAX_STEPS`.

**Colab:** the next cell mounts Google Drive. Set ``REPO_DIR`` to your clone path (default matches the training notebooks under ``agent/``). You can also set env ``BALATRO_AGENT_REPO`` before running that cell to skip editing.

**Local:** the mount step is skipped; set ``REPO_DIR`` to your local repo root.

**Actions:** this notebook always samples from the policy (**``Categorical.sample``** per slot for selection and execution), same as ``TrainCombat`` / ``TrainCombatMinimal`` / PPO—there is no argmax path here.

**Valid actions:** the env requires **1–5 cards** in ``selection``. **Invalid selection** (empty or more than five) still calls ``env.step``, but the snapshot is left unchanged and you get the invalid-action reward path (``INVALID_ACTION_REWARD`` in ``balatro_lite_gym/environment.py``). Sampling can occasionally draw all “don’t select”; the next loop iteration re-samples from the same state.


In [ ]:
import os
import sys

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # local: set REPO_DIR below

REPO_DIR = os.environ.get(
    "BALATRO_AGENT_REPO",
    "/content/drive/MyDrive/balatro-agent-project",
)
REPO_ROOT = REPO_DIR

os.chdir(REPO_DIR)

for p in (
    REPO_DIR,
    os.path.join(REPO_DIR, "balatro_lite_gym"),
):
    if p not in sys.path:
        sys.path.insert(0, p)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Run configuration (edit here)

Change **only** the next code cell to pick the policy type (full vs minimal), **checkpoint** (`.pt`), **snapshot** (`.json`), seed, step limit, and optional device. Run cells top to bottom after editing.

In [ ]:
# =============================================================================
# EDIT YOUR RUN HERE — policy, weights, snapshot, rollout
# =============================================================================

from pathlib import Path

# "full"  -> CombatPPOAgent      (e.g. checkpoints/combat_ppo_iter_*.pt)
# "minimal" -> MinimalCombatPPOAgent (e.g. checkpoints/minimal_combat_ppo_iter_*.pt)
AGENT_KIND = "minimal"

CKPT_PATH = Path(REPO_ROOT) / "checkpoints" / "minimal_combat_ppo_iter_1000.pt"
SNAPSHOT_PATH = Path(REPO_ROOT) / "snapshots" / "straight_in_hand_18.json"

SEED = None
MAX_STEPS = 10_000
DEVICE = None  # None -> cuda if available else cpu (applied in the next validation cell after imports)

In [ ]:
import numpy as np
import torch
from torch.distributions import Categorical

from agent.minimal_model import MinimalCombatPPOAgent
from agent.model import CombatPPOAgent
from agent.ppo import get_card_mask, mask_logits
from agent.ppo_config import PPOConfig
from defs import CARD_RANK_LABELS, CardRank
from env.game_simulator import GameSimulator
from env.lite_combat_env import adapt_lite_vector_obs, dict_to_tensors
from environment import MAX_HAND_LENGTH, _is_invalid_selection, _selected_indices
from util import rank_from_card_id, suit_from_card_id


In [ ]:
# Must match ``CardSuit`` int order: CLUBS=0, DIAMONDS=1, HEARTS=2, SPADES=3 (``defs/suit.py``).
_SUIT_GLYPHS = "♣♦♥♠"


def _nested_obs_add_batch_dim(obs_raw: dict) -> dict:
    out: dict = {}
    for k, v in obs_raw.items():
        if isinstance(v, dict):
            out[k] = {
                kk: np.asarray(vv, dtype=np.float32)[np.newaxis, ...]
                for kk, vv in v.items()
            }
        else:
            out[k] = np.asarray(v, dtype=np.float32)[np.newaxis, ...]
    return out


def _card_face(card_id: int) -> str:
    r = CardRank(rank_from_card_id(card_id))
    s = int(suit_from_card_id(card_id))
    return f"{CARD_RANK_LABELS[r]}{_SUIT_GLYPHS[s]}"


def _action_is_valid_env_transition(snap, act: dict) -> bool:
    """True if ``act`` would take a legal play/discard path (not invalid selection / illegal discard)."""
    try:
        indices = _selected_indices(act["selection"], len(snap.hand))
    except (TypeError, IndexError, ValueError):
        return False
    if _is_invalid_selection(indices):
        return False
    at = int(act["action_type"])
    if at == 1:
        return snap.play_remaining > 0
    if at == 0:
        return snap.discard_remaining > 0
    return False


def _action_from_checkpoint(
    agent: torch.nn.Module,
    obs_t: dict,
) -> tuple[np.ndarray, float, np.ndarray]:
    """Sample actions from the policy (same as PPO rollout). Return flat action, value, P(select) per slot."""
    with torch.no_grad():
        sel_logits, exec_logits, value = agent(obs_t)
    card_mask = get_card_mask(obs_t)
    sel_logits = mask_logits(sel_logits, card_mask)
    slot_p_select = torch.softmax(sel_logits, dim=-1)[0, :, 1].cpu().numpy()

    sel_dist = Categorical(logits=sel_logits)
    exec_dist = Categorical(logits=exec_logits)
    card_sels = sel_dist.sample()
    executions = exec_dist.sample()

    exec_bit = int(executions[0].item())
    action = np.concatenate(
        [card_sels[0].cpu().numpy().astype(np.int8), np.array([exec_bit], dtype=np.int8)],
        axis=0,
    )
    v = float(value[0].squeeze(-1).item())
    return action, v, slot_p_select


In [ ]:
# Resolve paths / device (edit policy + paths in **Run configuration** above)
if AGENT_KIND not in ("full", "minimal"):
    raise ValueError(f"AGENT_KIND must be 'full' or 'minimal', got {AGENT_KIND!r}")

ckpt_path = CKPT_PATH.resolve()
snap_path = SNAPSHOT_PATH.resolve()
assert ckpt_path.is_file(), f"checkpoint not found: {ckpt_path}"
assert snap_path.is_file(), f"snapshot not found: {snap_path}"

device = torch.device(
    DEVICE if DEVICE is not None else ("cuda" if torch.cuda.is_available() else "cpu")
)


In [ ]:
try:
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(ckpt_path, map_location=device)

cfg = ckpt.get("config")
if cfg is None:
    cfg = PPOConfig()
    print("warning: checkpoint has no 'config'; using default PPOConfig()", file=sys.stderr)

if AGENT_KIND == "minimal":
    agent = MinimalCombatPPOAgent(
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        dim_ff=cfg.dim_ff,
        dropout=cfg.dropout,
    ).to(device)
else:
    agent = CombatPPOAgent(
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        dim_ff=cfg.dim_ff,
        dropout=cfg.dropout,
    ).to(device)
agent.load_state_dict(ckpt["model_state_dict"])
agent.eval()

sim = GameSimulator.from_json(snap_path, seed=SEED)
obs_raw = sim.obs
info = sim.info

step = 1
consecutive_invalid = 0
print(
    f"checkpoint: {ckpt_path}  |  agent={AGENT_KIND}  |  snapshot: {snap_path.name}  |  {device}  |  sample (PPO)"
)
print("\n--- initial snapshot (before step 1) ---\n")
sim.print_snapshot(deck="summary")

while step <= MAX_STEPS:
    snap = info["snapshot"]
    print(f"\n--- step {step} ---")

    obs_t = dict_to_tensors(
        adapt_lite_vector_obs(_nested_obs_add_batch_dim(obs_raw)),
        device,
    )
    action, val, slot_p = _action_from_checkpoint(agent, obs_t)

    n_real = len(snap.hand)
    indices = [i for i in range(MAX_HAND_LENGTH) if int(action[i]) == 1]
    indices = [i for i in indices if i < n_real]
    exec_bit = int(action[MAX_HAND_LENGTH])
    verb = "PLAY" if exec_bit == 0 else "DISCARD"
    picked = [_card_face(snap.hand[i].card_id) for i in indices if i < len(snap.hand)]
    mean_ps = float(slot_p[:n_real].mean())
    print(
        f"  value {val:+.4f}  |  mean P(select) {mean_ps:.3f}  |  {verb}  slots={indices}  cards={picked}"
    )

    card_sel = action[:MAX_HAND_LENGTH].astype(np.int8)
    execution = int(action[MAX_HAND_LENGTH])
    action_type = 1 if execution == 0 else 0
    act = {"selection": card_sel, "action_type": int(action_type)}

    if len(indices) == 0:
        print(
            "(warn) Sampled no cards — env requires 1–5; this step is invalid (snapshot unchanged). "
            "Next loop iteration re-samples from the same state."
        )

    stepped = _action_is_valid_env_transition(snap, act)
    _obs, r, term, _trunc, step_info = sim.step_print(act, deck="summary")
    obs_raw, info = sim.obs, sim.info
    combat_won = bool(step_info.get("combat_won", False))
    step += 1

    if stepped:
        consecutive_invalid = 0
        print(f"  env reward: {float(r):+.4f}  |  terminated={term}")
        if float(r) < 0 and not term:
            print(
                "  (invalid selection or illegal discard)"
            )
        if term:
            print(
                "(hint) Episode over: win."
                if combat_won
                else "(hint) Episode over: loss."
            )
            break
    else:
        consecutive_invalid += 1
        print("  env reward: (invalid action — snapshot unchanged)  |  terminated=False")
        if consecutive_invalid >= 20:
            print(
                "(stopping) 20 consecutive invalid steps — e.g. repeated empty selection while P(select) is low, "
                "or play/discard exhausted. Check printed hints above."
            )
            break
else:
    print(f"\nStopped after MAX_STEPS={MAX_STEPS} without terminal.")


checkpoint: /content/drive/MyDrive/balatro-agent-project/checkpoints/minimal_combat_ppo_iter_1000.pt  |  agent=minimal  |  snapshot: flush_in_hand_18.json  |  cuda  |  sample (PPO)

--- initial snapshot (before step 1) ---


=== State ===
  round chips: 0 / target 300  (need 300 more)
  hand_size=8  hands_left=4  discards_left=4  blind_id=-1  (non-boss blind (Small/Big))

=== Hand ===
              │ 4♦ │ 7♥ │ 3♦ │ K♦ │ T♦ │ K♠ │ 9♦ │ 8♦
  ────────────┼────┼────┼────┼────┼────┼────┼────┼────
  slot        │ 0  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ 7 
  card        │ 4♦ │ 7♥ │ 3♦ │ K♦ │ T♦ │ K♠ │ 9♦ │ 8♦
  enhancement │    │    │    │    │    │    │    │   
  edition     │    │    │    │    │    │    │    │   
  debuff      │    │    │    │    │    │    │    │   

=== Jokers ===
  (none)

=== Draw pile ===
  cards in draw pile: 44
  ┌───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┬───┐
  │   │ A │ 2 │ 3 │ 4 │ 5 │ 6 │ 7 │ 8 │ 9 │ T │ J │ Q │ K │
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┼─